In [1]:
from __future__ import annotations

import hashlib
import json
import time
from typing import Any

print("stdlib ready — hashlib, json, time")

stdlib ready — hashlib, json, time


In [2]:
def meets_difficulty(digest: str, difficulty: int) -> bool:
    """Lab rule: hash must start with `difficulty` hex zeros."""
    if difficulty < 0:
        raise ValueError("difficulty must be non-negative")
    return digest.startswith("0" * difficulty)


examples = [
    ("00ab...", 2),
    ("000c...", 3),
    ("0abc...", 2),
]
for h, d in examples:
    # pad to look like a hex digest for the demo
    digest = h.replace("...", "") + "f" * (64 - len(h.replace("...", "")))
    print(f"d={d}: starts with {digest[:max(d, 4)]!r} → {meets_difficulty(digest, d)}")

d=2: starts with '00ab' → True
d=3: starts with '000c' → True
d=2: starts with '0abc' → False


In [3]:
def sha256_hex(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


GENESIS_PREV = "0" * 64
BLOCK_REWARD = 50.0


class Block:
    def __init__(
        self,
        index: int,
        transactions: list[Any],
        previous_hash: str,
        nonce: int = 0,
        timestamp: float | None = None,
    ) -> None:
        self.index = index
        self.timestamp = time.time() if timestamp is None else timestamp
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        payload = {
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce,
        }
        return sha256_hex(json.dumps(payload, sort_keys=True, separators=(",", ":")))

    def __repr__(self) -> str:
        return f"Block(index={self.index}, nonce={self.nonce}, hash={self.hash[:12]}...)"


class Blockchain:
    def __init__(self) -> None:
        self.chain: list[Block] = [
            Block(0, [{"note": "genesis"}], GENESIS_PREV, 0, 0.0)
        ]

    def tip(self) -> Block:
        return self.chain[-1]

    def append_block(self, block: Block, difficulty: int | None = None) -> None:
        tip = self.tip()
        if block.index != tip.index + 1:
            raise ValueError("bad index")
        if block.previous_hash != tip.hash:
            raise ValueError("bad previous_hash")
        if block.hash != block.compute_hash():
            raise ValueError("bad hash")
        if difficulty is not None and not block.hash.startswith("0" * difficulty):
            raise ValueError("difficulty not met")
        self.chain.append(block)

    def verify_chain(self, difficulty: int | None = None) -> bool:
        if self.chain[0].hash != self.chain[0].compute_hash():
            return False
        for i in range(1, len(self.chain)):
            cur, prev = self.chain[i], self.chain[i - 1]
            if cur.index != i:
                return False
            if cur.hash != cur.compute_hash():
                return False
            if cur.previous_hash != prev.hash:
                return False
            if difficulty is not None and not cur.hash.startswith("0" * difficulty):
                return False
        return True


bc0 = Blockchain()
print("genesis tip:", bc0.tip())
print("verify (no PoW yet):", bc0.verify_chain())

genesis tip: Block(index=0, nonce=0, hash=aa7fed24e783...)
verify (no PoW yet): True


In [4]:
def mine_block(block: Block, difficulty: int) -> tuple[Block, int, float]:
    """Mine until hash has `difficulty` leading hex zeros.

    Returns (block, attempts, elapsed_seconds).
    """
    if difficulty < 0:
        raise ValueError("difficulty must be non-negative")
    prefix = "0" * difficulty
    nonce = 0
    t0 = time.perf_counter()
    while True:
        block.nonce = nonce
        digest = block.compute_hash()
        if digest.startswith(prefix):
            block.hash = digest
            elapsed = time.perf_counter() - t0
            return block, nonce + 1, elapsed
        nonce += 1


# Quick smoke test at d=2
trial = Block(1, [{"demo": True}], GENESIS_PREV, nonce=0)
mined, attempts, secs = mine_block(trial, difficulty=2)
print("hash     :", mined.hash)
print("prefix OK:", mined.hash.startswith("00"))
print("nonce    :", mined.nonce)
print("attempts :", attempts)
print(f"seconds  : {secs:.4f}")

hash     : 00c5de69e5744c69973502f48cb7496152efa7855f5655dc2d854703ae5bcbd1
prefix OK: True
nonce    : 48
attempts : 49
seconds  : 0.0021


In [6]:
#Question 2


def difficulty_study(
    difficulties: tuple[int, ...] = (2, 3, 4),
) -> list[dict[str, Any]]:
    """Run mining at several difficulties; return rows for the A6 report table."""
    rows: list[dict[str, Any]] = []
    for d in difficulties:
        blk = Block(1, [{"payload": "study", "d": d}], GENESIS_PREV)
        mined, attempts, secs = mine_block(blk, d)
        rows.append(
            {
                "difficulty": d,
                "attempts": attempts,
                "seconds": secs,
                "hash": mined.hash,
            }
        )
        print(
            f"d={d}: attempts={attempts:8d}  "
            f"time={secs:8.4f}s  hash={mined.hash[:16]}..."
        )
    return rows


rows = difficulty_study((2, 3, 4))
print()
print(f"{'d':>3}  {'attempts':>10}  {'seconds':>10}  hash prefix")
print("-" * 52)
for row in rows:
    d = row["difficulty"]
    print(
        f"{d:3d}  {row['attempts']:10d}  {row['seconds']:10.4f}  "
        f"{row['hash'][:d]}…"
 )

d=2: attempts=      19  time=  0.0010s  hash=0032abf04ded8323...
d=3: attempts=     786  time=  0.0271s  hash=000f985b6afe29ee...
d=4: attempts=  125037  time=  3.5698s  hash=0000b41e359313d3...

  d    attempts     seconds  hash prefix
----------------------------------------------------
  2          19      0.0010  00…
  3         786      0.0271  000…
  4      125037      3.5698  0000…


In [7]:

#Question 3
def make_coinbase(miner_address: str, reward: float = BLOCK_REWARD) -> dict[str, Any]:
    """Simplified block-reward transaction."""
    return {
        "sender": "NETWORK",
        "recipient": miner_address,
        "amount": reward,
        "timestamp": time.time(),
        "type": "coinbase",
    }


cb = make_coinbase("MinerAlice")
print(cb)
assert cb["sender"] == "NETWORK"
assert cb["recipient"] == "MinerAlice"
assert cb["amount"] == 50.0
assert "timestamp" in cb
print("coinbase shape OK")

{'sender': 'NETWORK', 'recipient': 'MinerAlice', 'amount': 50.0, 'timestamp': 1789352158.4648461, 'type': 'coinbase'}
coinbase shape OK


In [8]:
def mine_and_append(
    chain: Blockchain,
    mempool: list[Any],
    miner_address: str,
    difficulty: int,
) -> tuple[Block, int, float]:
    """Create block with coinbase + mempool, mine it, append to chain."""
    tip = chain.tip()
    txs = [make_coinbase(miner_address)] + list(mempool)
    block = Block(tip.index + 1, txs, tip.hash)
    mined, attempts, secs = mine_block(block, difficulty)
    chain.append_block(mined, difficulty=difficulty)
    return mined, attempts, secs


bc = Blockchain()
mempool = [{"sender": "Alice", "recipient": "Bob", "amount": 10}]
mined, attempts, secs = mine_and_append(
    bc,
    mempool,
    miner_address="MinerAlice",
    difficulty=3,
)

print("mined hash :", mined.hash)
print("attempts   :", attempts)
print(f"seconds    : {secs:.4f}")
print("verify     :", bc.verify_chain(difficulty=3))
print("chain len  :", len(bc.chain))
print("reward tx  :", mined.transactions[0])
print("payment tx :", mined.transactions[1])

mined hash : 00094f5361e51b5cba56f1126223b228e82f2a5364e53b682ed21b737a8df272
attempts   : 1603
seconds    : 0.0693
verify     : True
chain len  : 2
reward tx  : {'sender': 'NETWORK', 'recipient': 'MinerAlice', 'amount': 50.0, 'timestamp': 1789352166.6577876, 'type': 'coinbase'}
payment tx : {'sender': 'Alice', 'recipient': 'Bob', 'amount': 10}


In [9]:
# Tamper after mining: hash no longer matches recomputation
tampered = Blockchain()
mined2, _, _ = mine_and_append(
    tampered,
    [{"sender": "A", "recipient": "B", "amount": 1}],
    miner_address="MinerAlice",
    difficulty=2,
)
mined2.transactions[1]["amount"] = 999  # mutate payment after PoW
# stored hash is stale — verify_chain catches it
print("after mutation, verify:", tampered.verify_chain(difficulty=2))

# Fresh block without mining — append should raise
bare = Blockchain()
unmined = Block(1, [{"x": 1}], bare.tip().hash)
try:
    bare.append_block(unmined, difficulty=2)
    print("UNEXPECTED: unmined block accepted")
except ValueError as err:
    print("append rejected unmined block:", err)

after mutation, verify: False
append rejected unmined block: difficulty not met
